# Python Basics for Data Engineers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/python-basics.ipynb)

*Generated from `content/python-basics.md` for Google Colab.* Markdown explanations are preserved, and fenced examples are converted into executable notebook cells wherever a Python kernel can run them. Some cells still require the right data, tables, credentials, Java runtime, Spark runtime, or cloud account before they can succeed.

## How to run

- Open this notebook in Google Colab using the badge above.
- Run the setup cell first.
- Run cells one by one and replace sample table names, file paths, credentials, and cloud resources with your own.
- SQL cells use `%%sql` with an in-memory SQLite connection unless you configure another database.
- PySpark cells need `pyspark` and Java. Colab usually has Java; install PySpark from the setup hint if missing.

Notebook inventory: **360 markdown cells**, **114 code cells**. Detected dependencies: standard library only.

In [ ]:
# Colab setup: run this cell first.
import importlib.util
import sys

required_packages = []
package_modules = {'apache-airflow': 'airflow', 'delta-spark': 'delta', 'google-cloud-storage': 'google.cloud.storage', 'ipython-sql': 'sql'}
print("Python:", sys.version.split()[0])

for package in required_packages:
    module = package_modules.get(package, package.split("-")[0])
    status = "available" if importlib.util.find_spec(module) else "missing"
    print(f"{package:22} {status}")

print("\nIf a required package is missing, run:")
print('# No pip installs detected.')

if "ipython-sql" in required_packages:
    try:
        get_ipython().run_line_magic("load_ext", "sql")
    except Exception:
        print("Install ipython-sql, restart the runtime if needed, then rerun this setup cell.")
    else:
        get_ipython().run_line_magic("sql", "sqlite://")
        print("SQL magic connected to an in-memory SQLite database.")

This guide teaches the Python you need before any data-engineering tool makes sense. Everything is explained in plain language, with a small runnable example for each idea and a short "Try it" exercise so you can check yourself.

How to use it:

+ Run one code cell at a time and read the output before moving on.
+ Change a value and run the cell again. Experimenting is the fastest way to learn.
+ Do the "Try it" exercises. They are short, and they build on the examples above them.
+ When a new term appears, it is explained in that same section.

You do not need programming experience to start. By the end you will be comfortable with variables, collections,
functions, classes, exceptions, files, dates and tests: the everyday vocabulary of pipeline code.

## How do values, variables, and dynamic typing work in Python?

A variable is a sticky label. You stick the label on a value. Later you can peel it off and stick it on a different value. The label itself has no type.

The value carries the type. `3` is an integer, `"three"` is a string, and the name `count` is just a label pointing at one of them. Because labels can move, Python is **dynamically typed**: it checks types while the program runs, not before it starts. The same name can point at a number now and a string later, and Python allows it.

Python code is built from two things. An **expression** produces a value, such as `2 + 3`. A **statement** performs an action, such as an assignment, an `if`, or a `def`. When you run an expression, Python hands you back a value.

In [ ]:
print(2 + 3)
print("orders".upper())
print(len(["raw", "bronze", "silver"]))

Two built-ins tell you what you are holding. `type(x)` gives the exact type. `isinstance(x, T)` asks whether `x` behaves like a `T`, or like a subtype of `T`. Use `type` when you care about the exact class and `isinstance` when you care about behavior.

In [ ]:
value = 3
print(type(value).__name__)
print(isinstance(value, int))
print(isinstance(True, int))

Rebinding a name to a value of a new type is legal:

In [ ]:
count = 3
count = "three"
rate: float = 0.75
print(count, rate)

`rate: float = 0.75` is a **type hint**, also called an annotation. It is a note for readers and for tools such as type checkers. Python itself does not enforce it while the program runs.

Two comparison operators look similar but ask different questions. `==` asks "do these have the same value?" `is` asks "are these the very same object?" The built-in `id()` returns an object's identity, which is a number unique to that object while the program runs.

In [ ]:
a = [1, 2]
b = [1, 2]
c = a

print(a == b)
print(a is b)
print(a is c)

`a` and `b` hold equal contents, so `==` is `True`. They are two separate lists, so `is` is `False`. `c` is another label on the same list as `a`, so `a is c` is `True`.

The everyday use of `is` is checking for `None`:

In [ ]:
value = None
print(value is None)
print(value == None)

Both lines print `True`, but `is None` is the style everyone expects.

### Try it

Make a list, point a second name at it, and compare the two names.

Hint: use `==` first, then `is`.

Expected result:

```text
True
True
```

The first line is `==` (same contents) and the second is `is` (same object).

Takeaway: names are labels, values have types, and `==` compares values while `is` compares identity.

## How do numbers and strings work in Python?

Numbers and text are the raw material of data work, so it pays to know exactly what Python does with them.

There are three number types you will use constantly. `int` holds whole numbers and can grow as large as memory allows. `float` holds decimal values approximately, because it stores them in binary. `Decimal` holds decimals exactly, which is why money code often uses it. **Immutable** means a value can never be changed after it is created. Strings are immutable.

Here are the arithmetic operators:

| Operator | Meaning | Example |
|---|---|---|
| `+` | add | `2 + 3` |
| `-` | subtract | `10 - 4` |
| `*` | multiply | `6 * 7` |
| `/` | true division | `7 / 2` gives `3.5` |
| `//` | floor division | `7 // 2` gives `3` |
| `%` | remainder | `7 % 2` gives `1` |
| `**` | power | `2 ** 10` |

In [ ]:
from decimal import Decimal

print(7 / 2, 7 // 2, 2**10)
print(divmod(7, 2))
print(0.1 + 0.2, Decimal("0.1") + Decimal("0.2"))

Two things to notice. `/` always returns a float, even when the division comes out even. `divmod(a, b)` returns the quotient and the remainder together as a tuple. The float line prints `0.30000000000000004` because `0.1` and `0.2` have no exact binary form. `Decimal` gets the exact answer because it stores base-10 digits.

For money, count integer cents or use `Decimal`:

In [ ]:
price_cents = 1999
quantity = 3
total_cents = price_cents * quantity

print(total_cents)
print(f"${total_cents / 100:.2f}")

Data from files and APIs usually arrives as text, so convert it before doing math:

In [ ]:
from decimal import Decimal

raw_amount = "10.50"
amount = Decimal(raw_amount)

print(amount * Decimal("2"))

A string is a **sequence**, which means an ordered collection you can index and slice. Picture a string as a row of numbered boxes, starting at `0`. The slice syntax is `[start:stop:step]`, and `stop` is not included.

In [ ]:
text = "python"

print(text[0])
print(text[-1])
print(text[1:4])
print(text[:2])
print(text[2:])

A negative index counts from the end, so `text[-1]` is the last character. `text[1:4]` gives positions 1, 2, and 3. A missing `start` means "from the beginning" and a missing `stop` means "to the end".

An **f-string** is the normal way to format values. Put an `f` before the quote and put expressions inside braces. A colon inside the braces adds a format spec, such as `:.2f` for two decimal places or `:>8` for right alignment in eight characters.

In [ ]:
order_id = "ORD-2026-000123"
region, year, seq = order_id.split("-")

print(f"{region} {year} {int(seq):,} {0.4217:.2%}")
print(order_id[:3], order_id[-6:], order_id[::-1])

`split("-")` breaks the string on each dash and returns a list. The assignment unpacks that list into three names. `order_id[::-1]` uses a step of `-1` to reverse the string.

String methods return new strings instead of changing the original, because strings are immutable:

In [ ]:
raw = "  ORD-001,paid  "
clean = raw.strip().lower()

print(repr(raw))
print(clean)
print(clean.startswith("ord"))
print(clean.split(","))

`strip()` removes leading and trailing whitespace, `lower()` lowercases, and `startswith()` returns a boolean. Chaining them applies each call to the result of the previous one.

Some data is not text at all. `bytes` holds raw binary data, which is a sequence of numbers rather than characters. Decode bytes to text with an encoding before treating them as a string, and encode text to bytes when you send it out.

In [ ]:
payload = "café".encode("utf-8")
text = payload.decode("utf-8")

print(payload)
print(text)

The first print shows `b'caf\xc3\xa9'`, the raw UTF-8 bytes. The second shows the text again. Always pass `encoding="utf-8"` when you cross the text and binary boundary.

### Try it

Split `"2026-10-05"` into three parts and print the year.

Hint: use `.split("-")` and index the result.

Expected result:

```text
2026
```

Takeaway: pick `int`, `float`, or `Decimal` on purpose, and remember that string methods hand you back a new string.

## What are the core collections: lists, tuples, sets, and dicts?

A collection holds many values in one object. Think of a `list` as a numbered shelf: the order matters and you can swap items in and out. A `tuple` is a shelf that is glued shut. A `set` is a bag that silently drops duplicates. A `dict` is a phone book: you look up a name and get a value back.

In data work you meet these everywhere: a list of rows, a tuple for one fixed pair, a set of IDs you have already seen, and a dict of configuration.

| Type | Simple meaning | Common use |
|---|---|---|
| `list` | ordered and changeable | rows, columns, batches |
| `tuple` | ordered and usually not changed | fixed pairs such as `(id, amount)` |
| `set` | unique values | membership checks and duplicate removal |
| `dict` | key-value lookup | records, configs, mappings |

In [ ]:
columns = ["order_id", "amount", "amount"]
unique_columns = set(columns)
row = ("A1", 10.5)
record = {"order_id": "A1", "amount": 10.5}

print(columns)
print(unique_columns)
print(row[0])
print(record["amount"])

Lists and tuples are both **ordered**, which means the items keep the position you gave them. Lists are **mutable**, which means you can change them in place. Tuples are immutable.

In [ ]:
stages = ["raw", "bronze"]
stages.append("silver")
stages[0] = "landing"

print(stages)
print(stages[-1])

`append` adds to the end, and assigning to `stages[0]` replaces the first item. Both change the same list.

Tuples fit fixed shapes, such as one row from a database driver:

In [ ]:
order = ("A1", 10.5)
order_id, amount = order

print(order_id)
print(amount)

That unpacking trick works for any sequence of the right length.

Sets keep only unique values and answer "is this in there?" quickly:

In [ ]:
seen_ids = {"A1", "A2"}

print("A1" in seen_ids)
print("A3" in seen_ids)
print({"A1", "A2"} | {"A2", "A3"})

The `|` operator is union, `&` is intersection, and `-` is difference:

In [ ]:
print({"id", "ts"} & {"id", "to"})
print({"id", "ts"} - {"id", "to"})

Dictionaries map keys to values. A key must be **hashable**, which means Python can turn it into a fixed number, called a hash, and use that number to find the value. Strings, numbers, and tuples are hashable. Lists and dicts are not, because they can change.

In [ ]:
order = {"order_id": "A1", "amount": 10.5}

print(order["order_id"])
print(order.get("status", "unknown"))

`order["order_id"]` raises `KeyError` when the key is missing, while `order.get("status", "unknown")` returns the default instead. The `**` operator merges dicts into a new dict:

In [ ]:
print({**{"id": 1}, "amount": 2})
print("amount" in {"amount": 2})

`Counter` and `defaultdict` save you from writing "if the key is missing, create it" by hand:

In [ ]:
from collections import Counter, defaultdict

rows = [("us", 12), ("eu", 7), ("us", 5)]

by_region = defaultdict(list)
for name, amount in rows:
    by_region[name].append(amount)

print(dict(by_region))
print(Counter(name for name, _ in rows).most_common(1))

`Counter` counts how often each item appears. `defaultdict(list)` creates an empty list the first time a key is used, so `append` always works.

One trap: methods that change a collection in place return `None`. `append`, `update`, and `sort` are all mutators.

In [ ]:
numbers = [3, 1, 2]
result = numbers.sort()

print(numbers)
print(result)

`numbers` is sorted and `result` is `None`. Use `sorted(numbers)` when you want a new sorted list and want to keep the original order.

### Try it

Count the unique values in `["a", "b", "a", "c", "b"]`.

Hint: wrap the list in `set(...)` and print its length.

Expected result:

```text
3
```

Takeaway: pick the container that matches the job, and remember that in-place methods return `None`.

## How do comprehensions work in Python?

A **comprehension** is a short way to build a new collection from an existing one. Instead of writing three lines of loop, you describe what you want in one line. It is a loop with the boilerplate removed.

A comprehension walks through an **iterable**, which is anything you can go through one item at a time, such as a list, tuple, set, dict, or string.

Read the pattern from left to right:

```text
[expression for item in iterable if condition]
```

The `for` part says where the values come from. The optional `if` part keeps only some of them. The expression at the front says what to put into the result.

Compare a normal loop with the comprehension that replaces it:

In [ ]:
rows = [{"id": 1, "amount": 10}, {"id": 2, "amount": 0}, {"id": 3, "amount": 25}]

paid_amounts = []
for row in rows:
    if row["amount"] > 0:
        paid_amounts.append(row["amount"])

print(paid_amounts)

In [ ]:
rows = [{"id": 1, "amount": 10}, {"id": 2, "amount": 0}, {"id": 3, "amount": 25}]

paid_amounts = [row["amount"] for row in rows if row["amount"] > 0]
print(paid_amounts)

Both print `[10, 25]`.

The brackets around the comprehension decide what you get back:

In [ ]:
rows = [{"id": 1, "amount": 10}, {"id": 2, "amount": 0}, {"id": 3, "amount": 25}]

ids = [row["id"] for row in rows]
lookup = {row["id"]: row["amount"] for row in rows}
unique = {row["amount"] for row in rows}
total = sum(row["amount"] for row in rows)

print(ids)
print(lookup)
print(sorted(unique))
print(total)

`[...]` builds a list, `{key: value ...}` builds a dict, `{...}` builds a set, and a bare `(...)` builds a **generator expression**. A generator expression is lazy: it produces values one at a time instead of building the whole collection in memory first, which is why `sum(...)` can take one directly.

You can stack `for` clauses to flatten data. The clauses run left to right, so the leftmost one is the outer loop:

In [ ]:
flat = [item for batch in [[1, 2], [3]] for item in batch]
print(flat)

Keep comprehensions boring. If you need temporary variables, logging, API calls, mutation, or error handling, a normal `for` loop is easier to read.

### Try it

Build a list of the squares of the even numbers in `range(10)`.

Hint: try `[x * x for x in range(10) if x % 2 == 0]`.

Expected result:

```text
[0, 4, 16, 36, 64]
```

Takeaway: a comprehension is a compact loop that builds a collection, and it should stay short enough to read in one line.

## What is the difference between mutability, references, copy, and deepcopy?

Assignment does not copy. When you write `alias = original`, you give the same object a second name. Both names point to one object in memory. A name that points at an object is called a **reference**.

This matters for **mutable** objects, which are objects you can change in place, such as lists and dicts. Change the object through one name and every other name sees the change.

In [ ]:
original = ["raw", "bronze"]
alias = original

alias.append("silver")
print(original)

The output is `['raw', 'bronze', 'silver']`, even though you never touched `original`. There is only one list with two labels on it. This situation is called **aliasing**.

The built-in `id()` shows an object's identity. It is useful for learning and debugging, but do not build logic on it.

In [ ]:
original = ["raw", "bronze"]
alias = original
copy_of_original = list(original)

print(id(original) == id(alias))
print(id(original) == id(copy_of_original))

The first line is `True` because both names point at one list. The second is `False` because `list(original)` builds a brand-new list.

To get an independent object, you copy it. A **shallow copy** copies only the outer container. A **deep copy** copies the outer container and every nested object inside it, all the way down.

In [ ]:
import copy

config = {"source": "s3", "options": {"retries": 3}}
shallow = copy.copy(config)
deep = copy.deepcopy(config)

shallow["options"]["retries"] = 99
deep["options"]["retries"] = 1

print(config["options"]["retries"])
print(deep["options"]["retries"])

The shallow copy still shares the inner `options` dict, so changing `retries` through it also changes `config`. The deep copy shares nothing, so `config` keeps its own value. Deep copy is safer but slower, so use it only when you need fully independent nested state.

The same rule explains why a function can change your list. Passing an argument passes a reference, not a copy:

In [ ]:
def mutate(items):
    items.append(0)

data = [1, 2]
mutate(data)
print(data)

The function prints nothing, but `data` is now `[1, 2, 0]`, because `items` is another name for the same list. If the function did `items = []` instead, it would only rebind its own local name and your list would be untouched.

### Try it

Make a shallow copy of `{"a": [1]}` and append `2` to the copied inner list. Then print the original.

Hint: `copy.copy` does not copy the inner list.

Expected result:

```text
[1, 2]
```

The original changed too, because the inner list is shared.

Takeaway: assignment aliases, a shallow copy shares nested objects, and a deep copy gives you a fully independent object.

## How does control flow work in Python?

Control flow decides which lines run and how many times they run. Python reads your code from top to bottom, and these statements let you skip, repeat, or choose.

The building blocks are `if` for a choice, `for` to walk through an iterable, `while` to repeat until something changes, `break` to leave a loop early, `continue` to jump to the next round, and `match` for tidier multi-case branching in Python 3.10 and later.

An `if` statement is just a yes-or-no question. Python evaluates the condition, gets `True` or `False`, and runs one block or the other. Indentation marks the block, so those four spaces matter.

In [ ]:
amount = 125

if amount > 100:
    print("large order")
else:
    print("small order")

A `for` loop takes each item from an iterable in turn:

In [ ]:
for stage in ["raw", "bronze", "silver"]:
    print("run", stage)

A `while` loop keeps going as long as its condition stays true. Make sure something inside changes, or it never stops:

In [ ]:
attempt = 1

while attempt <= 3:
    print("attempt", attempt)
    attempt += 1

Conditions use **truthiness**, which means Python treats some values as false even when they are not `False`. Those values are `False`, `None`, `0`, `0.0`, `""`, and empty containers such as `[]`, `{}`, and `set()`. Everything else counts as true.

In [ ]:
values = [10, 0, None, ""]

for value in values:
    if value:
        print("truthy:", value)
    else:
        print("falsy:", repr(value))

`repr(value)` shows a value in a form that reveals what it is, which is why the empty string prints as `''` instead of a blank line. Truthiness is handy for checks like `if not rows:`, which means "nothing came back".

`break` leaves a loop immediately, and `continue` skips the rest of the current round. A `for` loop can also have an `else` block, which runs only when the loop finished without hitting `break`:

In [ ]:
for part in ["2026-10-01", "2026-10-02"]:
    if part == "2026-10-02":
        print("found", part)
        break
else:
    print("missing")

`match` compares one value against several patterns. It reads like a series of cases, and `_` means "anything else". A guard is an extra `if` attached to a case:

In [ ]:
def classify(status, attempts):
    match status:
        case "retry" if attempts < 3:
            return "backoff"
        case "retry":
            return "dead-letter"
        case _:
            return "unknown"

print(classify("retry", 1))
print(classify("retry", 5))

### Try it

Loop over `[2, 4, 7, 8]` and print only the even numbers.

Hint: use `continue` when a number is odd.

Expected result:

```text
2
4
8
```

Takeaway: use `if` to choose, `for` and `while` to repeat, and `break` or `continue` to steer a loop.

## How do functions, arguments, scope, and closures work?

A function is a labelled machine. You feed it input, it does one small job, and it hands back output. You build the machine once with `def`, then run it as many times as you like.

The values you pass in are **arguments**. The names in the `def` line are **parameters**. The value the function hands back is its **return value**.

In [ ]:
def dollars(cents: int) -> float:
    return cents / 100

print(dollars(1250))

If a function never reaches a `return`, Python hands back `None`. `None` simply means "no value".

In [ ]:
def log_message(message: str) -> None:
    print("LOG:", message)

result = log_message("started")
print(result)

Parameters come in a few flavours. A normal parameter must be given a value. A **default parameter** supplies a fallback. `*args` scoops up extra positional values into a tuple. `**kwargs` scoops up extra keyword values into a dictionary. A parameter placed after a bare `*` can only be passed by name.

In [ ]:
def load(*path_parts: str, mode: str = "batch", **options: object) -> dict:
    return {"path": "/".join(path_parts), "mode": mode, "options": options}

print(load("s3://bucket", "orders", mode="stream", compression="zstd"))

Here is a trap. Python builds default values once, when it defines the function. So a mutable default such as `[]` is shared by every call.

In [ ]:
def remember_bad(value: str, bucket: list[str] = []) -> list[str]:
    bucket.append(value)
    return bucket

print(remember_bad("a"))
print(remember_bad("b"))

The fix is to default to `None` and build the list inside the function.

In [ ]:
def remember_good(value: str, bucket: list[str] | None = None) -> list[str]:
    bucket = [] if bucket is None else bucket
    bucket.append(value)
    return bucket

print(remember_good("a"))
print(remember_good("b"))

**Scope** is the part of a program where a name can be used. Python looks a name up in this order: first the current function, then any enclosing function, then the module (the global names), then the built-ins.

In [ ]:
count = 10          # global scope

def show() -> None:
    count = 99      # local scope: a different name
    print("inside:", count)

show()
print("outside:", count)

A **closure** is a function that remembers names from the function where it was created. The inner function keeps those names alive even after the outer function has finished.

In [ ]:
def make_counter(start: int = 0):
    count = start

    def increment(step: int = 1) -> int:
        nonlocal count
        count += step
        return count

    return increment

counter = make_counter(10)
print(counter(), counter(5), counter())

`nonlocal` says "change the `count` in the outer function" instead of creating a brand new local name.

### Try it

Write `make_multiplier(factor)`. It returns a function. That returned function takes a number and multiplies it by `factor`.

Hint: keep `factor` as a parameter of the outer function, then return an inner function.

Expected result: `double = make_multiplier(2)` and then `double(21)` prints `42`.

In [ ]:
def make_multiplier(factor: int):
    def multiply(value: int) -> int:
        return value * factor
    return multiply

double = make_multiplier(2)
print(double(21))

Why this matters for data work: small functions with clear arguments are easy to test and reuse in every pipeline.

**Takeaway:** A function is a machine. Scope decides which names it can see. A closure is a function that remembers the names where it was born.

## How do modules, packages, and the main guard work?

A **module** is one `.py` file. A **package** is a folder full of modules. **Importing** means using code from another file instead of copying it.

`import math` gives you the whole module object. You reach the names inside it with a dot, such as `math.sqrt`.

In [ ]:
import math

print(math.sqrt(16))

You can also pull out a single name with `from`.

In [ ]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))

A package is just a folder, and `__init__.py` marks it as one. A common layout looks like this.

```text
jobs/
  __init__.py
  parsers.py
  run.py
```

Here is a real package built in a temporary folder, so you can see the import work.

In [ ]:
import sys, tempfile
from pathlib import Path

project = Path(tempfile.mkdtemp())
(project / "jobs").mkdir()
(project / "jobs" / "__init__.py").write_text("")
(project / "jobs" / "parsers.py").write_text("def parse(raw):\n    return raw['id']\n")

sys.path.insert(0, str(project))
from jobs.parsers import parse
print(parse({"id": "A1"}))

The line that adds to `sys.path` was needed only because the demo package lives in a temporary folder. Real code should not do that.

When Python runs a file, it sets a built-in name called `__name__`. If you started the file directly, `__name__` is `"__main__"`. If another file imported it, `__name__` is the module name. That is what the **main guard** checks.

In [ ]:
if __name__ == "__main__":
    print("This runs only when you execute the file directly.")

The guard means "run this part only when the file is started on its own, not when another file imports it."

Prefer absolute imports such as `from jobs.parsers import parse`, keep `__init__.py` free of heavy work, and avoid changing `sys.path` from library code.

### Try it

Create a file `greet.py` that defines `def hello(name): return f"Hi {name}"`, then import it from another file and print `hello("Sam")`.

Hint: put both files in the same folder and run the second one. Expected result: the program prints `Hi Sam`.

In [ ]:
# greet.py would contain:
def hello(name: str) -> str:
    return f"Hi {name}"

print(hello("Sam"))

Why this matters for data work: pipelines are built from modules, so each transformation can live in one reusable file.

**Takeaway:** A module is a file, a package is a folder, and the main guard keeps import-time code from running when a file is imported.

## How do classes, objects, and methods work?

A **class** is a blueprint. An **object** is one thing built from that blueprint. You create an object by calling the class, like `Extractor("https://api.example.com")`.

Use a class when data and behaviour belong together. An API extractor has a URL, a page size, and a method that fetches a page.

Before classes, you might pass a plain dictionary around.

In [ ]:
extractor = {"url": "https://api.example.com/orders", "page_size": 100}
print(extractor["url"])

That is fine for tiny scripts. A class earns its place when the object also has behaviour, such as validating settings, fetching a page, closing a connection, or building itself from environment settings.

The special method `__init__` runs when an object is created and sets up its data. The first parameter of every instance method is `self`, which means "this particular object". A **method** is simply a function that lives inside a class.

In [ ]:
class Extractor:
    kind = "http"                  # class attribute: shared by all

    def __init__(self, url: str, page_size: int = 100) -> None:
        self.url = url             # instance attribute: one per object
        self.page_size = page_size

    def fetch(self, page: int) -> dict:
        return {"page": page, "url": self.url, "size": self.page_size}

extractor = Extractor("https://api.example.com/orders")
print(extractor.fetch(1), extractor.kind)

An **instance attribute** belongs to one object. A **class attribute** is shared by all objects, unless one object shadows it by assigning its own value.

In [ ]:
class Extractor:
    def __init__(self, url: str, page_size: int = 100) -> None:
        self.url = url
        self.page_size = page_size

small = Extractor("https://api.example.com/small", page_size=10)
large = Extractor("https://api.example.com/large", page_size=1000)
print(small.page_size, large.page_size)

A `@classmethod` receives the class itself, named `cls`, and is often used as an alternative constructor. A `@staticmethod` receives neither `self` nor `cls`; it is just a plain helper that lives in the class for tidiness.

In [ ]:
class Extractor:
    def __init__(self, url: str, page_size: int = 100) -> None:
        self.url = url
        self.page_size = page_size

    @classmethod
    def from_env(cls, url: str) -> "Extractor":
        return cls(url, page_size=50)

    @staticmethod
    def is_partition_column(column: str) -> bool:
        return column.startswith("dt")

print(Extractor.from_env("u").page_size, Extractor.is_partition_column("dt=2026-10-04"))

### Try it

Add a method `size_in_kb()` to a `Batch` class. It should return `len(self.rows)` times 512, divided by 1000.

Hint: store `rows` in `__init__` as `self.rows`. Expected result: `Batch([1, 2]).size_in_kb()` prints `1.024`.

In [ ]:
class Batch:
    def __init__(self, rows: list) -> None:
        self.rows = rows

    def size_in_kb(self) -> float:
        return len(self.rows) * 512 / 1000

print(Batch([1, 2]).size_in_kb())

Why this matters for data work: a class keeps a connection, its settings, and its methods in one testable place.

**Takeaway:** A class is a blueprint, an object is the thing built from it, and `self` means "this particular object".

## How do inheritance, super, polymorphism, and MRO work?

**Inheritance** lets one class reuse and specialise another class. Write `class ParquetSink(Sink)` and `ParquetSink` gets everything `Sink` has. Use inheritance only for a true **is-a** relationship: a `ParquetSink` is a kind of `Sink`.

**Polymorphism** means you call the same method on different objects and each object picks its own behaviour. Every sink below has `.write(...)`, but each one writes differently.

Sometimes you only want to share a helper. Then **composition** is simpler: one object holds another object as an attribute. This is a **has-a** relationship.

In [ ]:
class RetryPolicy:
    def __init__(self, attempts: int) -> None:
        self.attempts = attempts

class Loader:
    def __init__(self, retry_policy: RetryPolicy) -> None:
        self.retry_policy = retry_policy

loader = Loader(RetryPolicy(attempts=3))
print(loader.retry_policy.attempts)

A child class can replace a method from its parent. The parent here only promises that `write` exists.

In [ ]:
class Sink:
    def write(self, rows: list) -> int:
        raise NotImplementedError

class NullSink(Sink):
    def write(self, rows: list) -> int:
        return 0

sink = NullSink()
print(type(sink).__name__, sink.write([{}, {}]))

`super()` means "call the next implementation in the method resolution order". The **method resolution order**, or **MRO**, is the order Python searches classes for an attribute or method.

In [ ]:
class Sink:
    def __init__(self, path: str) -> None:
        self.path = path

class ParquetSink(Sink):
    def __init__(self, path: str, compression: str = "zstd") -> None:
        super().__init__(path)
        self.compression = compression

sink = ParquetSink("s3://lake/orders")
print(sink.path, sink.compression)
print([c.__name__ for c in ParquetSink.__mro__])

Because both children share the same method name, a loop can treat them the same way. That is polymorphism in action.

In [ ]:
class Sink:
    def write(self, rows: list) -> int:
        raise NotImplementedError

class ParquetSink(Sink):
    def write(self, rows: list) -> int:
        return len(rows) * 2

class NullSink(Sink):
    def write(self, rows: list) -> int:
        return 0

for sink in (ParquetSink(), NullSink()):
    print(type(sink).__name__, sink.write([{}, {}]))

Use composition when the relationship is "has-a": a `Loader` that holds an `S3Client` is easier to test than `Loader(S3Client)`. When you need to state a contract without sharing code, use a `Protocol` or an abstract base class.

### Try it

Make a `CsvSink` that inherits from `Sink` and returns `len(rows)` from `write`. Then print `CsvSink().write([{}, {}, {}])`.

Hint: define the parent `Sink` first, then `class CsvSink(Sink)`. Expected result: `3`.

In [ ]:
class Sink:
    def write(self, rows: list) -> int:
        raise NotImplementedError

class CsvSink(Sink):
    def write(self, rows: list) -> int:
        return len(rows)

print(CsvSink().write([{}, {}, {}]))

Why this matters for data work: one loop can write to Parquet, CSV, or nowhere, because every sink speaks the same `.write(...)` method.

**Takeaway:** Inherit for "is-a", hold objects for "has-a", and let `super()` and the MRO decide which method actually runs.

## What are the most important dunder methods?

A **dunder method** is a method whose name starts and ends with two underscores. "Dunder" just means "double underscore", so `__len__` is said as "dunder len".

You almost never call a dunder method yourself. Python calls it for you when you use normal syntax. When you write `len(batch)`, Python looks for `batch.__len__()`. When you write `for row in batch`, Python looks for `batch.__iter__()`.

The useful ones to know are: `print(obj)` uses `__str__`, `repr(obj)` uses `__repr__`, `len(obj)` uses `__len__`, `for item in obj` uses `__iter__`, and `obj == other` uses `__eq__`.

Before writing a class, ask yourself: "Do I want this object to behave like a collection, a value, a function, or a context manager?" The answer tells you which dunder method you need.

In [ ]:
class Batch:
    def __init__(self, rows: list[dict]) -> None:
        self._rows = rows

    def __repr__(self) -> str:
        return f"Batch(rows={len(self._rows)})"

    def __str__(self) -> str:
        return f"batch of {len(self._rows)} rows"

    def __len__(self) -> int:
        return len(self._rows)

batch = Batch([{"id": 1}, {"id": 2}])
print(batch, repr(batch), len(batch))

`__iter__` makes the object loopable, `__eq__` decides what "equal" means, and `__hash__` lets the object go into a set or act as a dictionary key. Objects that are equal must have the same hash.

In [ ]:
class Batch:
    def __init__(self, rows: list[dict]) -> None:
        self._rows = rows

    def __iter__(self):
        return iter(self._rows)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Batch) and self._rows == other._rows

    def __hash__(self) -> int:
        return hash(tuple(r["id"] for r in self._rows))

batch = Batch([{"id": 1}])
print(list(batch), batch == Batch([{"id": 1}]), len({batch, Batch([{"id": 2}])}))

Three more you will meet: `__getitem__` powers indexing and slicing, `__call__` makes an object callable like a function, and `__enter__` plus `__exit__` make an object usable in a `with` statement.

In [ ]:
class Doubler:
    def __call__(self, value: int) -> int:
        return value * 2

print(Doubler()(21))

Use dunder methods sparingly. They are powerful because they make your objects feel like built-in types, but surprising behaviour makes code hard to debug.

### Try it

Give a class `Bag` a `__len__` method that returns `len(self.items)`. Then print `len(Bag([1, 2, 3]))`.

Hint: the returned value must be an `int`. Expected result: `3`.

In [ ]:
class Bag:
    def __init__(self, items: list) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

print(len(Bag([1, 2, 3])))

Why this matters for data work: a `Batch` with `__len__` and `__iter__` can be used in the same loops and checks as a plain list.

**Takeaway:** Dunder methods are hooks that let Python syntax work on your own objects, and you implement them instead of calling them.

## How do properties and encapsulation work?

**Encapsulation** means keeping the inside details of an object tidy and hidden, so outside code uses a clear, small surface. Python does not have strict private fields. Instead it uses naming conventions.

A plain `name` is public data. A single underscore, `_name`, means "internal; please do not use this from outside the class". A double underscore, `__name`, triggers **name mangling**, which quietly renames the attribute, and it is rarely needed.

A **property** lets outside code use simple attribute syntax while your class still runs code behind the scenes. Think of `@property` as a controlled door: the caller reads `snapshot.rows`, but the class decides what happens.

Without a property, a class cannot stop bad data.

In [ ]:
class PlainSnapshot:
    def __init__(self, rows: int) -> None:
        self.rows = rows

plain = PlainSnapshot(-100)
print(plain.rows)

With a property, the class can reject bad values at the moment they are assigned.

In [ ]:
class Snapshot:
    def __init__(self, rows: int) -> None:
        self.rows = rows              # goes through the setter
    @property
    def rows(self) -> int:
        return self._rows
    @rows.setter
    def rows(self, value: int) -> None:
        if value < 0:
            raise ValueError("rows cannot be negative")
        self._rows = value

print(Snapshot(2048).rows)

A property with no setter is read-only. You can also compute a value on the fly, so it always matches the current data.

In [ ]:
class Snapshot:
    def __init__(self, rows: int) -> None:
        self._rows = rows

    @property
    def size_mb(self) -> float:
        return round(self._rows * 512 / 1_000_000, 3)

print(Snapshot(2048).size_mb)

Now check that the validation really fires.

In [ ]:
class Snapshot:
    def __init__(self, rows: int) -> None:
        self.rows = rows
    @property
    def rows(self) -> int:
        return self._rows
    @rows.setter
    def rows(self, value: int) -> None:
        if value < 0:
            raise ValueError("rows cannot be negative")
        self._rows = value
try:
    Snapshot(-1)
except ValueError as exc:
    print("validation worked:", exc)

Notice three things. `snapshot.rows` looks like a plain attribute read. `self.rows = rows` inside `__init__` calls the setter. And `size_mb` has no setter, so it can only be read.

Start with plain attributes. Add a property only when you need validation, a computed value, or a shim while refactoring.

### Try it

Give `Score` a property `points` that raises `ValueError` when the value is above `100`. Then print `Score(50).points`.

Hint: store the real value in `self._points` and add a `@points.setter`. Expected result: `50`.

In [ ]:
class Score:
    def __init__(self, points: int) -> None:
        self.points = points

    @property
    def points(self) -> int:
        return self._points

    @points.setter
    def points(self, value: int) -> None:
        if value > 100:
            raise ValueError("too high")
        self._points = value

print(Score(50).points)

Why this matters for data work: validation in one setter protects every row, instead of trusting every caller to check the same rule.

**Takeaway:** Python hides by convention, and `@property` gives you a plain-looking attribute with your own code behind it.

## How do dataclasses, NamedTuple, and enums work?

These three tools all solve the same problem: a plain dictionary is flexible, but too flexible. If you type `row["amunt"]` by mistake, Python does not complain until that exact line runs in production. A small named object gives your data a shape you can see and check.

Use `@dataclass` when you want a normal object with named fields and maybe some methods. Use `NamedTuple` when you want a tiny, unchangeable record that still unpacks like a tuple. Use `Enum` when only a few fixed values are allowed.

An **enum** is a small class whose members are the only legal values. It stops "magic strings" from spreading through your code, so a typo becomes an error instead of a quiet bug.

In [ ]:
from enum import StrEnum, auto

class Stage(StrEnum):
    EXTRACT = auto()
    LOAD = auto()

stage = Stage.EXTRACT
print(stage is Stage.EXTRACT, stage.value)

A **dataclass** is a class where Python writes the boring parts for you. You list the fields, and `@dataclass` generates `__init__`, a readable `__repr__`, equality comparison, and the field storage.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class S3Location:
    bucket: str
    key: str

    @property
    def uri(self) -> str:
        return f"s3://{self.bucket}/{self.key}"

print(S3Location("lake", "raw.json").uri)

`frozen=True` means "do not allow changes after creation." That is handy for config and keys you pass around. If you try to change a frozen field, Python raises an error right away:

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: int

point = Point(1)
try:
    point.x = 2
except Exception as exc:
    print(type(exc).__name__, exc)

A `NamedTuple` is a tuple with field names. It is immutable for free, and it still supports unpacking like `order_id, amount = row`. Reach for it when the value is a small record and you want tuple behaviour.

In [ ]:
from typing import NamedTuple

class Row(NamedTuple):
    order_id: str
    amount: float

row = Row("A1", 10.5)
order_id, amount = row
print(row == Row("A1", 10.5), amount)

Choose a dataclass when the object has behaviour or validation. Choose a `NamedTuple` when it is a small immutable record. Choose an enum when the value must come from a fixed list.

### Try it

Create a `Priority` enum with three members, then change the `Point` example so `point.x = 2` does not raise. Hint: drop `frozen=True` from the decorator. Expected result: the assignment works and `print(point.x)` shows `2`.

Why this matters for data work: a named record or enum turns a silent typo in a column name into a loud error before the job runs.

## How do exceptions and context managers work?

An **exception** is Python's way of saying "something went wrong here." Think of a fire alarm: the alarm itself does not fix anything, but it tells you exactly where to look, and you choose how to respond.

Do not try to avoid every error. Instead, expect errors at your boundaries: parsing user input, reading files, calling APIs, connecting to databases, converting strings to numbers, and looking up missing dictionary keys.

Use this shape:

+ `try`: the code that might fail.
+ `except`: your recovery or translation.
+ `else`: runs only when no exception happened.
+ `finally`: cleanup that always runs.

In [ ]:
try:
    amount = float("10.5")
except ValueError:
    amount = 0.0
else:
    print("parsed:", amount)
finally:
    print("cleanup runs either way")

Here is the same idea with input that fails:

In [ ]:
raw_amount = "not-a-number"

try:
    amount = float(raw_amount)
except ValueError as exc:
    print("could not parse amount:", exc)
    amount = 0.0

print(amount)

A **context manager** is the object behind the `with` keyword. It is a door that closes itself: it guarantees setup at the start and cleanup at the end, even if the code in the middle blows up. Files, database sessions, temporary tables, and locks are common examples.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.gettempdir()) / "context-demo.txt"

with path.open("w", encoding="utf-8") as handle:
    handle.write("hello")

print(path.read_text(encoding="utf-8"))

You can build your own context manager with the `@contextmanager` decorator. Put setup before `yield`, and put cleanup after it. Python always runs the cleanup, even when the block raises.

In [ ]:
from contextlib import contextmanager

@contextmanager
def staging_table(name: str):
    print("create temp table", name)
    try:
        yield name
    finally:
        print("drop temp table", name)

with staging_table("stg_orders") as table:
    print("using", table)

You also group your own errors into a small family. Callers can then catch every pipeline failure with one `except`:

In [ ]:
class PipelineError(Exception):
    """Base class for every pipeline failure."""

class TransientError(PipelineError):
    def __init__(self, status: int, message: str) -> None:
        super().__init__(message)
        self.status = status

try:
    raise TransientError(503, "service unavailable")
except PipelineError as exc:
    print("caught:", exc.status, exc)

Never write a bare `except:`. It also catches `KeyboardInterrupt` and `SystemExit`, which should usually stop your program. Catch `Exception` only at a top-level boundary that logs the failure. When you translate an error, keep the original with `raise ConfigError("bad config") from exc`.

### Try it

Write a function `to_int(text)` that raises `ValueError` with a helpful message when `text` is not a number, then catch it in a `try` block. Hint: put the field name in the message. Expected result: `could not parse batch_size='abc'`.

Why this matters for data work: errors that name the bad input save you an hour of guessing which row broke the job.

## How do iterators and generators work in Python?

An **iterable** is anything you can loop over. A list is iterable, a file handle is iterable, and a generator is iterable.

An **iterator** is the object that keeps track of where you are in a loop. You ask it for the next value, and it either hands one over or says it is finished.

A **generator** is a function that produces values lazily, one at a time. Imagine a vending machine: each press of the button gives you one item, and the machine remembers where it was. That is why a generator can handle a 100-million-row file without loading it all into memory.

A list is eager: it builds every value now. A generator is lazy: it produces one value when asked. Compare them:

In [ ]:
eager = [number * 2 for number in range(5)]
lazy = (number * 2 for number in range(5))

print(eager)
print(lazy)
print(next(lazy))
print(next(lazy))

`eager` prints as a real list because all values were built immediately. `lazy` prints as a generator object because the values are still waiting to be produced. `next(lazy)` asks for one value. Once a generator gives a value, it moves forward and never rewinds.

To write a generator, use `yield` instead of `return`. `return` gives one final result and stops the function. `yield` gives one value now and pauses the function until you ask for the next one.

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3

stream = numbers()
print(next(stream))
print(next(stream))
print(list(stream))

When a generator runs out of values, `next()` raises `StopIteration`. That is the signal a `for` loop uses to know it is done:

In [ ]:
stream = (value for value in [10, 20])

print(next(stream))
print(next(stream))
try:
    next(stream)
except StopIteration:
    print("no more values")

Generators chain together nicely, and `yield from` passes values along while staying lazy:

In [ ]:
from collections.abc import Iterator
def parse_lines(lines: list[str]) -> Iterator[dict]:
    for line in lines:
        if line.strip():
            key, _, value = line.partition("=")
            yield {"key": key, "value": int(value)}

def with_run_id(rows: Iterator[dict]) -> Iterator[dict]:
    yield {"key": "_run_id", "value": "run-7"}
    yield from rows
stream = with_run_id(parse_lines(["a=1", "", "b=2"]))
print(next(stream), list(stream), list(stream))

The last `list(stream)` is empty because the generator was already consumed. That is a common bug: if you need to read the data twice, turn it into a list first. If you need low memory, keep it a generator and process it once.

If you want to know what a generator is under the hood: it is an object with `__iter__` returning `self`, and `__next__` raising `StopIteration` when the values run out. You rarely write that by hand, but that is the whole protocol.

### Try it

Write a generator `countdown(n)` that yields `n, n-1, ..., 1`, then print `list(countdown(3))`. Hint: use a `while` loop with `yield` and decrease `n`. Expected result: `[3, 2, 1]`.

Why this matters for data work: generators let you stream a huge file row by row instead of loading it all into memory.

## How do decorators work in Python?

A **decorator** is a function that takes another function and returns a replacement function. Think of gift wrap: the present inside is unchanged, but you now hand over the wrapped box.

The `@` syntax is just short for calling the decorator with your function. This:

```text
@decorator
def fn():
    ...
```

means exactly this:

```text
fn = decorator(fn)
```

So after decoration, the name `fn` points at the replacement, not at your original function. That is how decorators add retries, logging, timing, caching, authentication, and validation without touching the function body.

In [ ]:
import functools
def announce(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print("calling", fn.__name__)
        return fn(*args, **kwargs)
    return wrapper

@announce
def add(a, b):
    return a + b
print(add(2, 3))

Here is what happened, step by step. `announce` received the original `add`. It created `wrapper`, which prints and then calls `add`. It returned `wrapper`. The name `add` now points to `wrapper`.

`functools.wraps` copies the original name and docstring onto the wrapper. Always use it, because logs, traces, and tests read that name.

A decorator that takes a setting, such as `@retry(attempts=3)`, needs one extra layer. Calling `retry(attempts=3)` returns a decorator, and that decorator then receives your function:

```text
@retry(attempts=3)  ->  retry(attempts=3) returns decorate
def fetch(): ...    ->  decorate(fetch)    returns wrapper
```

Here is a plain retry decorator in action. It is the same shape as `announce`, but it calls the function again when the function raises `TimeoutError`:

In [ ]:
def retry(fn):
    def wrapper(*args, **kwargs):
        for attempt in range(1, 4):
            try:
                return fn(*args, **kwargs)
            except TimeoutError:
                print("attempt", attempt, "failed")
    return wrapper
@retry
def fetch():
    raise TimeoutError("deadline")
print(fetch())

Read that from the inside out: `fetch()` is the business function, `@retry` replaces it with `wrapper`, and `wrapper` calls the real `fetch`, catches the transient error, and tries again. This short version drops `functools.wraps` to save space; keep it in real code, as `announce` did above. Also re-raise the error after the last attempt instead of falling through to `None`, so a broken job fails loudly.

Python also ships some decorators for you. `functools.lru_cache` remembers the result of each call, so the second call with the same argument is free:

In [ ]:
import functools

@functools.lru_cache(maxsize=128)
def region_for(code: str) -> str:
    print("looking up", code)
    return {"us": "americas", "de": "emea"}[code]

region_for("us")
region_for("us")
print(region_for.cache_info())

In real backoff, add jitter and a time cap, and retry only errors that are actually transient.

### Try it

Write a decorator `shout` that prints `before` and `after` around any function call, then apply it to a function that prints `hi`. Hint: copy the `announce` shape. Expected result: `before`, then `hi`, then `after`.

Why this matters for data work: one retry or logging decorator can protect every API call in a pipeline with no copy-paste.

## What functional tools does Python provide?

Python has small built-in helpers that shorten common loop patterns. They are not magic. You can always rewrite each one as a plain `for` loop, and doing that once is a good way to learn them.

The ones you will use most in data code are `enumerate`, `zip`, `any`, `all`, and `sorted`.

`enumerate` gives you a position and a value at the same time, so you never need a manual counter:

In [ ]:
columns = ["order_id", "amount", "status"]

for position, column in enumerate(columns, start=1):
    print(position, column)

`zip` pairs values from two iterables and stops at the shorter one:

In [ ]:
names = ["order_id", "amount"]
values = ["A1", 10.5]

print(dict(zip(names, values)))

`any` is true when at least one value is true. `all` is true when every value is true. Both stop as soon as the answer is known, so they can be fast:

In [ ]:
amounts = [10, 0, 25]

print(any(amount == 0 for amount in amounts))
print(all(amount >= 0 for amount in amounts))

`map` applies a function to every item, and `filter` keeps the items that pass a test. `reduce` folds a list into a single value. They all exist, but comprehensions are often easier to read:

In [ ]:
from functools import reduce
from operator import add

rows = [{"region": "us", "amount": 10}, {"region": "eu", "amount": 5}]
amounts = list(map(lambda row: row["amount"], rows))
big = list(filter(lambda row: row["amount"] > 5, rows))

print(amounts, [row["region"] for row in big], reduce(add, amounts, 0))

`sorted` returns a new ordered list, while `list.sort()` changes the list in place. A key function can return a tuple for multiple fields, and negating a number sorts it in descending order:

In [ ]:
rows = [{"region": "us", "amount": 10}, {"region": "eu", "amount": 25}]

print(sorted(rows, key=lambda row: (-row["amount"], row["region"])))

### Try it

Given `pairs = [("a", 1), ("b", 2)]`, build the dictionary `{"a": 1, "b": 2}` with `zip` and check whether any value is greater than 1. Hint: use `dict(zip(...))` and `any(...)`. Expected result: `{'a': 1, 'b': 2}` and `True`.

Why this matters for data work: these helpers replace manual counters and index juggling, which is where off-by-one bugs live.

## How do type hints and mypy work?

A **type hint** (also called an **annotation**) is a note that says what kind of value a function expects and returns, for example `def add(a: int, b: int) -> int`. It is documentation that tools can read.

Here is the important part: Python does not enforce type hints at runtime. The hints are not checked when your program runs. This function happily accepts strings:

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

print(add(1, 2))
print(add("a", "b"))

So why write them? Because a **static type checker** such as `mypy` or `pyright` reads your code without running it and warns you about mismatches before production. The hints also help your editor autocomplete and help the next reader understand the shape of your data.

A good signature tells the reader a lot in one line:

In [ ]:
def total_amount(records: list[dict[str, float]]) -> float:
    return sum(record["amount"] for record in records)

print(total_amount([{"amount": 10.5}, {"amount": 2.0}]))

Reading it: `records` should be a list, each item should be a dictionary with string keys and float values, and the function returns a float.

Put hints on public functions, data structures, and boundaries where JSON, CSV, APIs, or database rows enter your code. Two helpers cover most data shapes. `TypedDict` describes the keys of a dictionary payload, so tools know exactly which keys to expect:

In [ ]:
from collections.abc import Iterable, Iterator
from typing import TypedDict

class Record(TypedDict):
    order_id: str
    amount: float

def totals(records: Iterable[Record]) -> Iterator[tuple[str, float]]:
    for record in records:
        yield record["order_id"], record["amount"]

print(list(totals([{"order_id": "A1", "amount": 10.5}])))

`Protocol` is for **structural typing**: it says "anything with this method counts", even if the class never inherits from it. That lets a test double stand in for a production class:

In [ ]:
from typing import Protocol

class Writer(Protocol):
    def write(self, text: str) -> int: ...

class FakeWriter:
    def write(self, text: str) -> int:
        return len(text)

writer: Writer = FakeWriter()
print(writer.write("hello"))

Run a checker from a terminal or a CI job, not usually from a notebook:

```text
uv run mypy --strict src/
uv run pyright src/
```

Use `TypedDict` when you still want plain dictionaries but want tools to understand the expected keys. Use a dataclass when you want an object with behaviour, validation, or methods.

### Try it

Add hints to `def double(value)` so it takes an `int` and returns an `int`, then write a call that a checker would flag, such as `double("a")`. Hint: the checker complains even though Python runs the code fine. Expected result: `mypy` reports a type error on the string argument.

Why this matters for data work: hints catch wrong column types at review time instead of at 2 a.m. when the pipeline fails.

## How do datetime and time zones work in Python?

A `datetime` object is one exact moment in time. A time zone is not a different moment. It is only a clock setting for showing that moment where you live. Noon in Tokyo and 10 PM in Chicago can be the very same instant, written two different ways.

The shared world clock is called **UTC** (Coordinated Universal Time). Store your timestamps in UTC. Compare them in UTC. Change to a local time zone only when a human needs to read the time, or when a business calendar says so.

Python has two kinds of `datetime` objects. A **naive** one has no time zone attached. It does not know where in the world it is. An **aware** one carries a `tzinfo` (time-zone info), so it knows its offset from UTC. Use aware datetimes in real pipeline code. Use `date` when you mean a calendar day, and `datetime` when you mean a precise instant.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

utc = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
new_york = utc.astimezone(ZoneInfo("America/New_York"))

print(utc.isoformat())
print(new_york.isoformat())
print(utc == new_york)

`utc` and `new_york` are the same instant, so the last line prints `True`. They just show different clock times.

Durations are not moments. Keep them as `timedelta` objects for as long as you can.

In [ ]:
from datetime import date, datetime, timedelta, timezone

utc = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)

print(utc.date())
print((utc + timedelta(days=1)).isoformat())
print(int(utc.timestamp()))
print((date(2026, 10, 5) - date(2026, 10, 4)).days)

`timestamp()` is the number of seconds since 1 January 1970 UTC, the "Unix epoch". Date math gives you whole days.

Avoid `datetime.utcnow()` and plain `datetime.now()`. They give you a naive value that quietly means "whatever the computer's clock says". Write `datetime.now(timezone.utc)` instead. The classic bug shows up when "today" depends on the time zone: midnight UTC is still yesterday in New York.

### Try it

- Task: take `datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)` and print its hour in three time zones.
- Hint: `astimezone(ZoneInfo("Asia/Tokyo")).hour`.
- Expected result: three different hours, all from the same instant.

Why this matters for data work: a wrong time zone silently moves rows into the wrong partition or the wrong day.

## How do file I/O, pathlib, JSON, and CSV work?

A **path** is the address of a file or folder. `pathlib.Path` turns that address into an object you can join, inspect, and read, instead of gluing strings together by hand.

Every file task has three steps. Pick a path. Read or write text with an encoding. Then parse that text into Python objects. Do not mix the steps. `encoding="utf-8"` keeps text readable for everyone. When you use the `csv` module, also pass `newline=""` so line endings stay correct.

JSON stores one value, often a nested record. **Newline-delimited JSON** (NDJSON) stores one complete record per line, which is much easier to stream.

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "note.txt"
    path.write_text("hello\n", encoding="utf-8")
    print(path.exists())
    print(path.read_text(encoding="utf-8").strip())

`tempfile.TemporaryDirectory()` deletes its folder when the `with` block ends, so nothing is left behind.

In [ ]:
import json

record = {"order_id": "A1", "items": ["book", "pen"]}
payload = json.dumps(record)
back = json.loads(payload)

print(payload)
print(back["items"][0])

`dumps` means "dump to string". `loads` means "load from string". The round trip gives you back a normal Python dictionary.

In [ ]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    with path.open("w", encoding="utf-8") as handle:
        for record in [{"id": 1}, {"id": 2}]:
            handle.write(json.dumps(record) + "\n")
    print(path.read_text(encoding="utf-8").splitlines())

CSV is plain text too, and every value comes back as a `str`. Convert the numbers yourself.

In [ ]:
import csv
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.csv"
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, ["order_id", "amount"])
        writer.writeheader()
        writer.writerow({"order_id": "A1", "amount": 10.5})
    row = next(csv.DictReader(path.open(newline="", encoding="utf-8")))
    print(row["amount"], type(row["amount"]).__name__)

JSON cannot store `Decimal` or `datetime` on its own. Add `default=str` to turn those into text.

In [ ]:
import json
from datetime import date
from decimal import Decimal

record = {"when": date(2026, 10, 4), "amount": Decimal("10.50")}

print(json.dumps(record, default=str))

### Try it

- Task: write two NDJSON lines to a temporary file, read them back, and print the number of records.
- Hint: `splitlines()` gives you one string per line.
- Expected result: `2`.

Why this matters for data work: most landing zones are just files, so clean path and encoding habits prevent corrupt reads.

## How do the GIL, threads, processes, and asyncio compare?

**Concurrency** means work overlaps in time. **Parallelism** means work truly runs at the same moment on more than one CPU core. They are not the same thing.

**CPython** is the standard Python you install from python.org. It has a **GIL**, the Global Interpreter Lock: a single microphone that every thread must share. Only one thread may run Python bytecode at a time. That sounds bad, but it is fine for waiting. When a thread waits on the network, S3, a database, or an API, it steps away from the microphone and lets others speak.

Two more words. A **thread** is a worker inside your one Python process, sharing all its memory. A **process** is a whole separate Python program with its own memory and its own GIL. A **coroutine** is a function that can pause at an `await` and let other coroutines run.

| Workload | Usual tool |
|---|---|
| Waiting on network or files | threads |
| Heavy CPU work | processes |
| Huge number of network calls | asyncio |

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch(url: str) -> str:
    time.sleep(0.01)   # pretend the network is slow
    return url.upper()

with ThreadPoolExecutor(max_workers=4) as pool:
    print(list(pool.map(fetch, ["a", "b"])))

Both calls wait at the same time, so the whole job takes about 0.01 seconds instead of 0.02.

In [ ]:
import asyncio

async def afetch(url: str) -> str:
    await asyncio.sleep(0.01)   # a coroutine may pause here
    return url.upper()

async def main() -> list[str]:
    return await asyncio.gather(*(afetch(u) for u in ["a", "b"]))

print(asyncio.run(main()))

`asyncio.gather` starts both coroutines and waits for both. In a normal `.py` script you run them with `asyncio.run(main())`. In a Jupyter notebook the kernel already has an event loop, so you just write `await main()`.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def square(n: int) -> int:
    return n * n

if __name__ == "__main__":   # needed on macOS and Windows
    with ProcessPoolExecutor(max_workers=2) as pool:
        print(list(pool.map(square, [1, 2, 3])))

Use `ProcessPoolExecutor` when Python itself is doing the math in a tight loop. The function and its data must be picklable, meaning Python can turn them into bytes and send them to another process. Use `asyncio.to_thread` to push an unavoidable blocking call off the event loop.

The simple test: if your program waits on something outside Python, threads may help. If Python is busy calculating, threads will not help because of the GIL.

### Try it

- Task: run `fetch("a")` and `fetch("b")` through a `ThreadPoolExecutor` and print the results.
- Hint: `pool.map(fetch, ["a", "b"])` returns a lazy iterator, so wrap it in `list()`.
- Expected result: `['A', 'B']`.

Why this matters for data work: pipelines spend most of their time waiting on storage and APIs, so the right tool turns hours into minutes.

## How do virtual environments, pip, uv, pyproject, and pytest work?

A **virtual environment** is a clean toolbox for one project. Each project gets its own set of packages, so installing something for project A never breaks project B.

`venv` creates that toolbox. `pip` puts packages into it. `uv` is a much faster tool that does the same jobs plus lockfiles, which record the exact versions you installed. `pyproject.toml` is the label on the box: it lists your project's name, its Python version, and its dependencies. `pytest` runs your tests.

Without a virtual environment, every project installs into the same global Python. One project upgrades `pydantic`, and another project stops working. That is the whole reason for the toolbox.

These are terminal commands. They are shown for reference, not run here.

```text
python -m venv .venv && source .venv/bin/activate
pip install -e ".[dev]"
python -m pytest -q
```

In [ ]:
%%bash
uv venv
uv sync --frozen
uv run pytest -q

Here is a small `pyproject.toml`. Dependencies are the packages your code needs. The `dev` extra holds tools you only need while developing.

```toml
[project]
name = "orders-pipeline"
requires-python = ">=3.11"
dependencies = ["httpx>=0.27", "pydantic>=2.7"]

Tests are just functions that check one behavior each. In a real project, `pytest` finds every function whose name starts with `test_` and runs it. A failing assertion should tell you exactly what changed.

In [ ]:
def normalize(value: str) -> str:
    return value.strip().lower()

def test_normalize() -> None:
    assert normalize(" Us ") == "us"
    assert normalize("EU") == "eu"

test_normalize()
print("checks passed")

### Try it

- Task: write a function `is_positive(n)` and one `test_` function that checks both a true and a false case.
- Hint: use `assert is_positive(3)` and `assert not is_positive(-1)`.
- Expected result: running the file prints nothing and exits with status 0.

Why this matters for data work: a pinned environment means the job that passed on your laptop also passes in CI and in production.

## What are the most common Python pitfalls?

A pitfall is a small mistake that looks harmless and causes a strange bug later. Five of them show up again and again in code review: mutable defaults, late-binding closures, float equality, string building in loops, and shadowing builtins. Each one has a one-line fix, so do not memorize them all at once. Learn the symptom, then learn the fix.

| Pitfall | Symptom | Fix |
|---|---|---|
| mutable default | values leak between calls | use `None` then create inside |
| late-binding closure | lambdas all see the last loop value | bind with `i=i` |
| float equality | decimal math surprises you | use `Decimal` or `math.isclose` |
| string building in loops | slow large string creation | collect pieces and `"".join(...)` |
| shadowing builtins | `list` or `sum` stops working | use names like `values`, `total` |

A **mutable default** is a default value like `[]` that is created once, when Python reads the function. That one list is then reused on every call.

In [ ]:
def add_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_bad("a"))
print(add_bad("b"))

The second call still remembers `"a"`, because both calls share the same list. The fix is to default to `None` and build a fresh list inside the function.

A **closure** is a function that remembers names from the code around it. A lambda in a loop remembers the variable, not its value at that moment, so every lambda sees the final value.

In [ ]:
funcs = [lambda: i for i in range(3)]
fixed = [lambda i=i: i for i in range(3)]

print([fn() for fn in funcs])
print([fn() for fn in fixed])

The first list is `[2, 2, 2]`. The second is `[0, 1, 2]`, because `i=i` copies the current value in as a default.

Floats cannot store most decimals exactly. So compare them with a small tolerance instead of `==`.

In [ ]:
import math

print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))

Adding strings in a loop makes a brand-new string every time. Collect the pieces and join them once.

In [ ]:
parts = [str(i) for i in range(20000)]

joined = "".join(parts)
print(len(joined), joined.endswith("19999"))

**Shadowing a builtin** means reusing a name Python already uses, such as `list`, `sum`, or `dict`. After that, the builtin stops working in that scope.

In [ ]:
values = [1, 2, 3]
total = sum(values)
print(total)

A few more to avoid: changing a list while you loop over it, using `is` for value comparison instead of `==` (`is` is for `None`), a bare `except:` that hides every error, and `dict.get` that hides a key you actually needed. Often `d["k"]` raising `KeyError` is the better signal.

### Try it

- Task: fix a function `collect(item, bucket=[])` so two calls return `["a"]` and `["b"]`.
- Hint: default to `None`, then `bucket = [] if bucket is None else bucket`.
- Expected result: the first call does not appear in the second call's list.

Why this matters for data work: these bugs produce wrong numbers, not crashes, and wrong numbers are the expensive kind.

## Real world use cases for Python basics in data engineering

### 1. A reusable API client class with retries and a context manager

Suppose your extractor calls an API and the network hiccups halfway through the run. If every script writes its own retry loop, each one fails in a different way and you cannot tell what happened. A small client class fixes that: the retry rule lives in one place, and the connection closes automatically when the `with` block ends.

In [ ]:
import time
class ApiClient:
    def __init__(self, attempts=3):
        self.attempts, self.calls, self.session = attempts, 0, None
    def __enter__(self):
        self.session = "open"
        return self
    def __exit__(self, exc_type, exc, tb):
        self.session = None
        return False
    def get(self, path):
        for attempt in range(1, self.attempts + 1):
            self.calls += 1
            if self.calls < self.attempts:   # first tries would be a 503
                time.sleep(0.01)
                continue
            return {"path": path, "attempt": attempt}

with ApiClient() as client:
    print(client.get("/orders"))
print("session:", client.session)

- `__enter__` opens the session and `__exit__` closes it, so cleanup happens even if the body raises an exception.
- Returning `False` from `__exit__` lets errors keep traveling up, which is what you usually want.
- The retry loop stays inside `get`, so calling code is just `client.get("/orders")`.

### 2. A validated config dataclass

Suppose a job reads its bucket name, table name, and batch size from environment variables. A typo like `BATCH_SIZE=0` may not blow up until hours into a load. Put the settings in one frozen dataclass and check them once, before any work starts.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PipelineConfig:
    source_bucket: str
    target_table: str
    batch_size: int = 50_000
    def validate(self):
        if self.batch_size <= 0:
            raise ValueError("batch_size must be positive: " + str(self.batch_size))

config = PipelineConfig("lake-raw", "analytics.orders", 25_000)
config.validate()
print(config.source_bucket, config.target_table, config.batch_size)

try:
    PipelineConfig("lake-raw", "analytics.orders", 0).validate()
except ValueError as err:
    print("rejected:", err)

- `validate` raises right away, so a bad value is caught at startup instead of during a long load.
- `frozen=True` stops you from reassigning fields by accident after the config is built.
- The whole config is one object, which is easy to pass into functions and easy to fake in a test.

### 3. A generator-based streaming reader

Suppose a finance export is 40 GB of newline-delimited JSON and the machine has 8 GB of RAM. Building a list of every row will run out of memory. Generators handle one line at a time, so the data flows through without ever being fully loaded.

In [ ]:
import json

def rows(lines):
    for line in lines:
        if line.strip():
            row = json.loads(line)
            if row["amount"] >= 0:
                yield row

def batches(rows, size):
    batch = []
    for row in rows:
        batch.append(row)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

lines = ['{"order_id": "A1", "amount": 10.5}', '{"order_id": "A2", "amount": -1}', '{"order_id": "A3", "amount": 4.0}']
for batch in batches(rows(lines), 2):
    print([row["order_id"] for row in batch])

- Nothing is read until the `for` loop asks for a batch, so memory stays near one batch.
- Each generator is a separate step, so you can test `rows` on three hardcoded lines and never touch a 40 GB file.
- The final `if batch:` keeps the last, partly filled batch from being dropped.

### 4. A retry decorator with exponential backoff

Suppose several loaders all need to survive the same flaky timeout. Copying the retry loop into each one makes them drift apart. A decorator puts the rule in one place and wraps any function you point it at.

In [ ]:
import time
def retry(fn):
    def wrapper(*args, **kwargs):
        for attempt in range(1, 4):
            try:
                return fn(*args, **kwargs)
            except TimeoutError:
                if attempt == 3:
                    raise
                time.sleep(0.01 * 2 ** (attempt - 1))
    return wrapper
calls = [0]

@retry
def load(name):
    calls[0] += 1
    if calls[0] < 3:
        raise TimeoutError(name)
    return "loaded:" + name

print(load("dt=2026-10-04"), "calls:", calls[0])

- The wrapper tries three times, waiting 0.01s and then 0.02s, so a brief outage does not kill the run.
- `load` contains only the load logic, and the retry rule is reused by pasting `@retry` on another function.
- Note that this simple version does not use `functools.wraps`, so the wrapped function loses its name and docstring.

### 5. A context manager for staging tables

Suppose a load needs a temp table that must be dropped even when the merge fails. If you drop it by hand at the end of the function, one early `return` or exception leaves it behind. A context manager guarantees the cleanup step runs no matter how the block exits.

In [ ]:
from contextlib import contextmanager

class FakeWarehouse:
    def __init__(self):
        self.log = []
    def execute(self, sql):
        self.log.append(sql)

@contextmanager
def staging_table(wh, name, source):
    wh.execute("CREATE TEMP TABLE " + name + " AS SELECT * FROM " + source)
    try:
        yield name
        wh.execute("MERGE INTO analytics.orders USING " + name)
    finally:
        wh.execute("DROP TABLE IF EXISTS " + name)

wh = FakeWarehouse()
with staging_table(wh, "stg_orders", "raw.orders") as staging:
    wh.execute("INSERT INTO " + staging + " SELECT * FROM raw.orders")
print(len(wh.log), wh.log[-1])

- Code before `yield` runs on entry, and code in `finally` runs on exit, even if the block raises.
- The merge sits after `yield`, so it only runs when the body finishes successfully.
- Every statement is appended to `FakeWarehouse.log`, which lets you assert the exact SQL order in a test.